## Közös havi ingestion futtatás

Ez a cella a kiválasztott brókerek pipeline-jait futtatja havi bontásban.

Ha `START_MONTH` és `END_MONTH` értéke `None`, akkor a futás a `config/download_period.csv` szerinti időszakot használja.  
Ha megadjuk őket `YYYY-MM` formátumban, akkor csak az adott teljes, lezárt hónapok kerülnek feldolgozásra.

A futás hónaponként halad: az adott hónap kiválasztott Binance itemei után az adott hónap kiválasztott Dukascopy itemei, majd az adott hónap kiválasztott Saxo Bank itemei futnak. Ezután lép a következő hónapra.

A `BROKERS` paraméterrel szabályozható, mely brókerek fussanak:
- `None`: minden támogatott bróker
- `["binance"]`: csak Binance
- `["dukascopy"]`: csak Dukascopy
- `["saxo_bank"]`: csak Saxo Bank
- `["binance", "dukascopy"]`: Binance és Dukascopy
- `["binance", "dukascopy", "saxo_bank"]`: minden jelenleg bekötött bróker

Az `ASSETS` paraméterrel szabályozható, mely assetek fussanak:
- `None`: minden, amit az adott bróker támogat a config szerint
- `["BTCUSD"]`: csak BTCUSD
- `["XAUUSD", "EURUSD"]`: csak a megadott assetek

Ha egy bróker nem támogat egy megadott assetet, arra nem jön létre futtatandó item.

Saxo Bank futtatásához a következő cellában meg kell adni egy érvényes, időszakos Saxo access tokent. A tokent nem mentjük `.env` fájlba és nem commitoljuk GitHubra; notebookban futás előtt kell frissen megadni.


In [1]:
from getpass import getpass

SAXO_ACCESS_TOKEN = getpass("Saxo access token: ")
SAXO_BASE_URL = "https://gateway.saxobank.com/sim/openapi"

In [3]:
from pipelines.monthly_ingestion_runner import run_monthly_ingestion_from_config


# Ha mindkettő None, akkor a config/download_period.csv szerinti időszak fut.
# Ha megadod őket, csak a megadott teljes hónapok futnak.
START_MONTH = "2024-07"
END_MONTH = "2025-12"

# Példa explicit időszakra:
# START_MONTH = "2024-02"
# END_MONTH = "2024-03"

# Futtatandó brókerek.
# None = minden támogatott bróker
# ["binance"] = csak Binance
# ["dukascopy"] = csak Dukascopy
# ["binance", "dukascopy"] = mindkettő
BROKERS = ["saxo_bank"]

# Futtatandó assetek.
# None = minden configban támogatott asset
# ["BTCUSD"] = csak BTCUSD
ASSETS = None

results_df = run_monthly_ingestion_from_config(
    start_month=START_MONTH,
    end_month=END_MONTH,
    brokers=BROKERS,
    assets=ASSETS,
    interval="1m",
    saxo_access_token=SAXO_ACCESS_TOKEN,
    saxo_base_url=SAXO_BASE_URL,
    saxo_print_progress=True,
)

results_df



=== 2024-07 ===
Binance: nincs futtatando item.
Dukascopy: nincs futtatando item.
Saxo Bank indul...
  XAUUSD 2024-07-01 00:00 downloading...
  XAUUSD 2024-07-01 00:00 ok rows=780
  XAUUSD 2024-07-01 12:00 downloading...
  XAUUSD 2024-07-01 12:00 ok rows=720
  XAUUSD 2024-07-02 00:00 downloading...
  XAUUSD 2024-07-02 00:00 ok rows=780
  XAUUSD 2024-07-02 12:00 downloading...
  XAUUSD 2024-07-02 12:00 ok rows=720
  XAUUSD 2024-07-03 00:00 downloading...
  XAUUSD 2024-07-03 00:00 ok rows=780
  XAUUSD 2024-07-03 12:00 downloading...
  XAUUSD 2024-07-03 12:00 ok rows=720
  XAUUSD 2024-07-04 00:00 downloading...
  XAUUSD 2024-07-04 00:00 ok rows=780
  XAUUSD 2024-07-04 12:00 downloading...
  XAUUSD 2024-07-04 12:00 ok rows=720
  XAUUSD 2024-07-05 00:00 downloading...
  XAUUSD 2024-07-05 00:00 ok rows=780
  XAUUSD 2024-07-05 12:00 downloading...
  XAUUSD 2024-07-05 12:00 ok rows=720
  XAUUSD 2024-07-06 00:00 downloading...
  XAUUSD 2024-07-06 00:00 ok rows=780
  XAUUSD 2024-07-06 12:00 dow

,status,broker,asset,broker_symbol,year,month,message,errors
0,uploaded,saxo_bank,XAUUSD,XAUUSD,2024,7,Raw chart and bronze OHLCV uploaded successfully.,[]
1,uploaded,saxo_bank,XAGUSD,XAGUSD,2024,7,Raw chart and bronze OHLCV uploaded successfully.,[]
2,uploaded,saxo_bank,EURUSD,EURUSD,2024,7,Raw chart and bronze OHLCV uploaded successfully.,[]
3,uploaded,saxo_bank,XAUUSD,XAUUSD,2024,8,Raw chart and bronze OHLCV uploaded successfully.,[]
4,uploaded,saxo_bank,XAGUSD,XAGUSD,2024,8,Raw chart and bronze OHLCV uploaded successfully.,[]
5,uploaded,saxo_bank,EURUSD,EURUSD,2024,8,Raw chart and bronze OHLCV uploaded successfully.,[]
6,uploaded,saxo_bank,XAUUSD,XAUUSD,2024,9,Raw chart and bronze OHLCV uploaded successfully.,[]
7,uploaded,saxo_bank,XAGUSD,XAGUSD,2024,9,Raw chart and bronze OHLCV uploaded successfully.,[]
8,uploaded,saxo_bank,EURUSD,EURUSD,2024,9,Raw chart and bronze OHLCV uploaded successfully.,[]
9,uploaded,saxo_bank,XAUUSD,XAUUSD,2024,10,Raw chart and bronze OHLCV uploaded successfully.,[]


## Binance havi letöltés

Ez a cella a Binance pipeline-t futtatja a megadott teljes hónapokra.

Ha nincs megadva `start_month` és `end_month`, akkor a `config/download_period.csv` szerinti időszak futna.  
Ha megadjuk őket, akkor csak az adott lezárt hónapok kerülnek feldolgozásra.

Fontos: csak teljes, múltbeli hónap tölthető le. Az aktuális hónap nem engedélyezett!


In [3]:
from pipelines.binance_runner import run_binance_from_config

import pandas as pd

# Letöltendő teljes hónapok:
# Kezdő hónap: 2024-01
# Záró hónap: 2024-03
#
# A pipeline:
# - Binance BTCUSDT 1 perces havi ZIP fájlokat használ
# - raw CSV-t ment/feltölt
# - bronze OHLCV parquet fájlt készít/feltölt
# - manifestet és _SUCCESS markert ír
# - meglévő _SUCCESS esetén skipeli az adott hónapot

results = run_binance_from_config(
    start_month="2024-01",
    end_month="2024-03",
    interval="1m",
)

pd.DataFrame([result.__dict__ for result in results])

,status,broker,asset,broker_symbol,year,month,message
0,skipped,binance,BTCUSD,BTCUSDT,2024,1,_SUCCESS already exists in Azure.
1,skipped,binance,BTCUSD,BTCUSDT,2024,2,_SUCCESS already exists in Azure.
2,skipped,binance,BTCUSD,BTCUSDT,2024,3,_SUCCESS already exists in Azure.


## Dukascopy havi letöltés

Ez a cella a Dukascopy pipeline-t futtatja a megadott teljes hónapokra.

Ha nincs megadva `start_month` és `end_month`, akkor a `config/download_period.csv` szerinti időszak futna.  
Ha megadjuk őket, akkor csak az adott lezárt hónapok kerülnek feldolgozásra.

Fontos: csak teljes, múltbeli hónap tölthető le.  
Az aktuális hónap nem engedélyezett.

A Dukascopy pipeline órás `.bi5` tick fájlokat tölt le, ezekből raw tick parquetet, majd bronze OHLCV parquetet készít. Emiatt egy teljes hónap több assetre hosszabb ideig futhat.


In [4]:
from pipelines.dukascopy_runner import run_dukascopy_from_config

import pandas as pd

# Letöltendő teljes hónapok:
# Kezdő hónap: 2024-01
# Záró hónap: 2024-01
#
# A pipeline:
# - Dukascopy órás .bi5 tick fájlokat használ
# - raw tick parquet fájlt készít/feltölt
# - bronze OHLCV parquet fájlt készít/feltölt
# - manifestet és _SUCCESS markert ír
# - meglévő _SUCCESS esetén skipeli az adott hónap/asset párt
#
# Fejlesztési próba:
# start_index=5, limit=1 csak a 2024-01 BTCUSD itemet futtatja.
#
# 2024-01 Dukascopy itemek:
# 0 = XAUUSD
# 1 = XAGUSD
# 2 = EURUSD
# 3 = US500
# 4 = DAX
# 5 = BTCUSD

results = run_dukascopy_from_config(
    start_month="2024-01",
    end_month="2024-01",
    interval="1m",
    start_index=5,
    limit=1,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

pd.DataFrame([result.__dict__ for result in results])


,status,broker,asset,broker_symbol,year,month,message,errors
0,skipped,dukascopy,BTCUSD,BTCUSD,2024,1,_SUCCESS already exists in Azure.,[]


## Saxo Bank direkt letöltés

Ez a cella csak a Saxo Bank pipeline-t futtatja.

A Saxo Bank adatok a `chart/v3/charts` API-ból érkeznek.  
A lekérés teljes hónapokra működik: ha `START_MONTH` és `END_MONTH` meg van adva, akkor csak a megadott lezárt hónapok futnak. Ha mindkettő `None`, akkor a `config/download_period.csv` szerinti időszak kerül feldolgozásra.

A pipeline minden hónap/asset párra:
- lekéri a Saxo Bank chart adatokat;
- egy napot két átfedő részletben kér le: `00:00-13:00` és `12:00-00:00`;
- az átfedésből származó duplikált timestamp sorokat kiszedi;
- raw chart parquet fájlt ír/feltölt;
- bronze OHLCV parquet fájlt ír/feltölt;
- manifestet és `_SUCCESS` markert ír;
- meglévő `_SUCCESS` esetén skipeli az adott hónap/asset párt.

A Saxo Bank FX/metal chart adatban nincs valódi volumen, ezért a bronze OHLCV `volume` mezője `NaN`.

A Saxo API használatához access token kell.  
Ezt nem tesszük `.env` fájlba, mert rövid életű token, hanem a következő cellában `getpass(...)` segítségével adjuk meg.

Fejlesztési próba esetén a `start_index` és `limit` paraméterekkel lehet csak néhány itemet futtatni.

2024-01 Saxo Bank itemek:
- `0 = EURUSD`
- `1 = XAUUSD`
- `2 = XAGUSD`


In [ ]:
from pipelines.saxo_bank_runner import run_saxo_bank_from_config

import pandas as pd

# Letöltendő teljes hónapok:
# Kezdő hónap: 2024-01
# Záró hónap: 2024-01
#
# A pipeline:
# - Saxo Bank chart/v3/charts API-t használ
# - egy napot két átfedő részletben kér le:
#   - 00:00-13:00
#   - 12:00-00:00
# - az átfedésből származó duplikált timestamp sorokat kiszedi
# - raw chart parquet fájlt készít/feltölt
# - bronze OHLCV parquet fájlt készít/feltölt
# - manifestet és _SUCCESS markert ír
# - meglévő _SUCCESS esetén skipeli az adott hónap/asset párt
#
# Fontos:
# - Saxo FX/metal chart adatban nincs valódi volumen
# - ezért a bronze OHLCV volume mező NaN lesz
#
# Fejlesztési próba:
# start_index=0, limit=1 csak az első Saxo itemet futtatja.
#
# 2024-01 Saxo Bank itemek:
# 0 = EURUSD
# 1 = XAUUSD
# 2 = XAGUSD

results = run_saxo_bank_from_config(
    access_token=SAXO_ACCESS_TOKEN,
    base_url=SAXO_BASE_URL,
    start_month="2024-01",
    end_month="2024-01",
    interval="1m",
    start_index=0,
    limit=1,
    timeout_sec=30,
    max_attempts=2,
    retry_sleep_sec=10,
    print_progress=True,
)

pd.DataFrame([result.__dict__ for result in results])
